In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Load datasets after cleaning.
customers   = pd.read_csv(r"../data/processed/customers_cleaned.csv")
products    = pd.read_csv(r"../data/processed/products_cleaned.csv")
payments    = pd.read_csv(r"../data/processed/payments_cleaned.csv")
geolocation = pd.read_csv(r"../data/processed/geolocation_cleaned.csv")
sellers     = pd.read_csv(r"../data/processed/sellers_cleaned.csv")
order_items = pd.read_csv(r"../data/processed/order_items_cleaned.csv",parse_dates=["shipping_limit_date"])
reviews     = pd.read_csv(r"../data/processed/reviews_cleaned.csv",parse_dates=["review_answer_timestamp","review_creation_date"])
orders      = pd.read_csv(r"../data/processed/orders_cleaned.csv",parse_dates=["order_purchase_timestamp","order_approved_at","order_delivered_customer_date","order_estimated_delivery_date","order_delivered_carrier_date"])

# Customers Analysis

In [3]:
# Count the number of unique customers.
customers["customer_unique_id"].nunique()

96096

In [4]:
# Merge orders table with customers table using customer_id in orders_customers table.
orders_customers = orders.merge(customers,how="left",on="customer_id")

In [5]:
# Find the customer with the highest number of orders.
orders_customers["customer_unique_id"].value_counts().head(1)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
Name: count, dtype: int64

In [6]:
# Find the top 10 customers by number of orders.
orders_customers["customer_unique_id"].value_counts().head(10)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
ca77025e7201e3b30c44b472ff346268     7
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
12f5d6e1cbf93dafd9dcc19095df0b3d     6
de34b16117594161a6a89c50b289d35a     6
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
f0e310a6839dce9de1638e0fe5ab282a     6
dc813062e0fc23409cd255f7f53c7074     6
Name: count, dtype: int64

In [7]:
# Merge orders_customers table with order_items table using order_id in orders_customers_orderItems table.
orders_customers_orderItems = orders_customers.merge(order_items,how="left",on="order_id")

In [8]:
# Find the customer with the highest total spending excluding freight charges.
customer_total_spending_without_fv = orders_customers_orderItems.groupby("customer_unique_id")["price"].sum()
print(f"Customer id : {customer_total_spending_without_fv.idxmax()}")
print(f"Total spending excluding freight charges : {customer_total_spending_without_fv.max()}")

Customer id : 0a0a92112bd4c708ca5fde585afaa872
Total spending excluding freight charges : 13440.0


In [9]:
# Find the customer with the highest total spending including freight charges.
customer_total_spending_with_fv = orders_customers_orderItems.groupby("customer_unique_id")[["price","freight_value"]].sum()
customer_total_spending_with_fv["total_spending"] = customer_total_spending_with_fv["price"] + customer_total_spending_with_fv["freight_value"]
print(f"Customer id : {customer_total_spending_with_fv['total_spending'].idxmax()}")
print(f"Total spending including freight charges : {customer_total_spending_with_fv['total_spending'].max()}")

Customer id : 0a0a92112bd4c708ca5fde585afaa872
Total spending including freight charges : 13664.08


In [10]:
# One-time vs Repeat Customers.
one_time_customers = (orders_customers["customer_unique_id"].value_counts()).values == 1
repeat_customers = (orders_customers["customer_unique_id"].value_counts()).values > 1
print(f"Number of one-time customers : {one_time_customers.sum()}")
print(f"Number of repeat customers : {repeat_customers.sum()}")

Number of one-time customers : 93099
Number of repeat customers : 2997


"One-time customers" dominate the total customer base, accounting for 96.88% (93,099 customers), whereas "repeat customers" represent a mere   3.12% (2,997 out of a total of 96,096 customers).

This significant gap indicates an over-reliance on customer acquisition costs (CAC) to generate revenue, coupled with a notable weakness in customer retention and the maximization of long-term customer lifetime value (CLV).

# Order Analysis

In [11]:
# Find the total number of orders.
print(f"The total number of orders is : {orders["order_id"].count() : ,}")

The total number of orders is :  99,441


In [12]:
# Count the number of orders in each order status.
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [13]:
# Count the number of orders by purchase year.
orders["purchase_year"].value_counts()

purchase_year
2018    54011
2017    45101
2016      329
Name: count, dtype: int64

In [14]:
# Find the month with the highest number of orders.
print(f"The month with the highest number of orders is : {orders.groupby("purchase_month_name")["order_id"].count().idxmax()}")

The month with the highest number of orders is : August


In [15]:
# Count the number of orders by purchase month name.
orders_by_month = orders.groupby("purchase_month_name")["order_id"].count()

In [16]:
# Find the top 3 months with the highest number of orders.
print(orders_by_month.nlargest(3))

purchase_month_name
August    10843
May       10573
July      10318
Name: order_id, dtype: int64


In [17]:
# Find the top 3 months with the lowest number of orders.
print(orders_by_month.nsmallest(3))

purchase_month_name
September    4305
October      4959
December     5674
Name: order_id, dtype: int64


# Sales Analysis

In [18]:
# Find the total sales excluding freight charges.
print(f"Total Sales : {order_items["price"].sum()}")

Total Sales : 13591643.7


In [19]:
# Find the mean of orders spending excluding freight charges.
print(f"The mean of orders is : {(order_items.groupby("order_id")["price"].sum()).mean()}")

The mean of orders is : 137.75407637889444


In [20]:
# Find the highest order value excluding freight charges.
print(f"The highest order value is : {(order_items.groupby("order_id")["price"].sum()).idxmax()}, and its value is :"
      f"{(order_items.groupby("order_id")["price"].sum()).max()}")

The highest order value is : 03caa2c082116e1d31e67e9ae3700499, and its value is :13440.0


# Product Analysis

In [21]:
# Find the Best-selling product.
order_items["product_id"].value_counts().head(1)

product_id
aca2eb7d00ea1a7b8ebd4e68314663af    527
Name: count, dtype: int64

In [22]:
# Find Highest-revenue product.
(order_items.groupby("product_id")["price"].sum()).idxmax()

'bb50f2e236e5eea0100680137654686c'

In [23]:
# Merge order_items table with products table using product_id in orderItems_products table.
orderItems_products = order_items.merge(products,on="product_id",how="left")

In [24]:
# Find Best-selling category.
orderItems_products["product_category_name"].value_counts().head(1)

product_category_name
cama_mesa_banho    11115
Name: count, dtype: int64

In [25]:
# Find Highest-revenue category.
(orderItems_products.groupby("product_category_name")["price"].sum()).idxmax()

'beleza_saude'

In [26]:
# Find Top 10 Best-selling Products.
order_items["product_id"].value_counts().head(10)

product_id
aca2eb7d00ea1a7b8ebd4e68314663af    527
99a4788cb24856965c36a24e339b6058    488
422879e10f46682990de24d770e7f83d    484
389d119b48cf3043d311335e499d9c6b    392
368c6c730842d78016ad823897a372db    388
53759a2ecddad2bb87a079a1f1519f73    373
d1c427060a0f73f6b889a5c7c61f2ac4    343
53b36df67ebb7c41585e8d54d6772e08    323
154e7e31ebfa092203795c972e5804a6    281
3dd2a17168ec895c781a9191c1e95ad7    274
Name: count, dtype: int64

In [27]:
# Top 10 Highest-revenue Products.
(order_items.groupby("product_id")["price"].sum()).nlargest(10)

product_id
bb50f2e236e5eea0100680137654686c    63885.00
6cdd53843498f92890544667809f1595    54730.20
d6160fb7873f184099d9bc95e30376af    48899.34
d1c427060a0f73f6b889a5c7c61f2ac4    47214.51
99a4788cb24856965c36a24e339b6058    43025.56
3dd2a17168ec895c781a9191c1e95ad7    41082.60
25c38557cf793876c5abdd5931f922db    38907.32
5f504b3a1c75b73d6151be81eb05bdc9    37733.90
53b36df67ebb7c41585e8d54d6772e08    37683.42
aca2eb7d00ea1a7b8ebd4e68314663af    37608.90
Name: price, dtype: float64

In [28]:
# Find Top 10 Best-selling Categories.
orderItems_products["product_category_name"].value_counts().head(10)

product_category_name
cama_mesa_banho           11115
beleza_saude               9670
esporte_lazer              8641
moveis_decoracao           8334
informatica_acessorios     7827
utilidades_domesticas      6964
relogios_presentes         5991
telefonia                  4545
ferramentas_jardim         4347
automotivo                 4235
Name: count, dtype: int64

In [29]:
# Find Top 10 Highest-revenue Categories.
(orderItems_products.groupby("product_category_name")["price"].sum()).nlargest(10)

product_category_name
beleza_saude              1258681.34
relogios_presentes        1205005.68
cama_mesa_banho           1036988.68
esporte_lazer              988048.97
informatica_acessorios     911954.32
moveis_decoracao           729762.49
cool_stuff                 635290.85
utilidades_domesticas      632248.66
automotivo                 592720.11
ferramentas_jardim         485256.46
Name: price, dtype: float64

# Seller Analysis

In [30]:
# Find number of orders per seller.
order_items["seller_id"].value_counts()

seller_id
6560211a19b47992c3666cc44a7e94c0    2033
4a3ca9315b744ce9f8e9374361493884    1987
1f50f920176fa81dab994f9023523100    1931
cc419e0650a3c5ba77189a1882b7556a    1775
da8622b14eb17ae2831f4ac5b9dab84a    1551
                                    ... 
a1bea9970d8f65b240555f99c2214f7f       1
d1aa1ec0839dcab73a6161130eb1f94a       1
dbc51f5e45d654ecc16cb68e6817ecea       1
3fefda3299e6dfaea3466ef346a3571a       1
80ceebb4ee9b31afb6c6a916a574a1e2       1
Name: count, Length: 3095, dtype: int64

In [31]:
# Find Top seller by number of orders.
order_items["seller_id"].value_counts().head(1)

seller_id
6560211a19b47992c3666cc44a7e94c0    2033
Name: count, dtype: int64

In [32]:
# Find Top seller by revenue.
# Revenue = Product Price only, excluding freight charges.
(orderItems_products.groupby("seller_id")["price"].sum()).idxmax()

'4869f7a5dfa277a7dca6462dcf3b52b2'

In [33]:
# Find Top 10 sellers by number of orders.
order_items["seller_id"].value_counts().head(10)

seller_id
6560211a19b47992c3666cc44a7e94c0    2033
4a3ca9315b744ce9f8e9374361493884    1987
1f50f920176fa81dab994f9023523100    1931
cc419e0650a3c5ba77189a1882b7556a    1775
da8622b14eb17ae2831f4ac5b9dab84a    1551
955fee9216a65b617aa5c0531780ce60    1499
1025f0e2d44d7041d6cf58b6550e0bfa    1428
7c67e1448b00f6e969d365cea6b010ab    1364
ea8482cd71df3c1969d7b9473ff13abc    1203
7a67c85e85bb2ce8582c35f2203ad736    1171
Name: count, dtype: int64

In [34]:
# Find Top 10 Sellers by revenue.
(orderItems_products.groupby("seller_id")["price"].sum()).nlargest(10)

seller_id
4869f7a5dfa277a7dca6462dcf3b52b2    229472.63
53243585a1d6dc2643021fd1853d8905    222776.05
4a3ca9315b744ce9f8e9374361493884    200472.92
fa1c13f2614d7b5c4749cbc52fecda94    194042.03
7c67e1448b00f6e969d365cea6b010ab    187923.89
7e93a43ef30c4f03f38b393420bc753a    176431.87
da8622b14eb17ae2831f4ac5b9dab84a    160236.57
7a67c85e85bb2ce8582c35f2203ad736    141745.53
1025f0e2d44d7041d6cf58b6550e0bfa    138968.55
955fee9216a65b617aa5c0531780ce60    135171.70
Name: price, dtype: float64

# Delivery Analysis

In [35]:
# Find Average approval time excluding orders with missing approval time.
valid_orders = orders.dropna()
average_approval_time = (valid_orders["order_approved_at"] - valid_orders["order_purchase_timestamp"]).mean()
average_approval_time_in_hours = round(average_approval_time.total_seconds()/(60*60),2)
print(f"Average approval time : {average_approval_time_in_hours} hour")

Average approval time : 10.28 hour


In [36]:
# Find Average shipping preparation time excluding orders with missing approval time or missing delivered carrier date.
valid_orders = orders.dropna(subset=["order_approved_at","order_delivered_carrier_date"])
average_shipping_preparation_time = (valid_orders["order_delivered_carrier_date"] - valid_orders["order_approved_at"]).mean()
average_shipping_preparation_time_in_hours = round(average_shipping_preparation_time.total_seconds()/(60*60*24),2)
print(f"Average shipping preparation time : {average_shipping_preparation_time_in_hours} days")

Average shipping preparation time : 2.81 days


In [37]:
# Find Average delivery time from the carrier to the customer excluding orders with missing delivered customer date or missing delivered carrier date.
valid_orders = orders.dropna(subset=["order_delivered_customer_date", "order_delivered_carrier_date"])
average_delivery_time = ((valid_orders["order_delivered_customer_date"] - valid_orders["order_delivered_carrier_date"]).mean())
average_delivery_time_in_days = round(average_delivery_time.total_seconds()/(24*60*60),2)
print(f"{average_delivery_time_in_days} days.")

9.33 days.


In [38]:
# Find Average order delivery time from purchase to customer delivery excluding orders with missing delivered customer date or missing purchase timestamp.
valid_orders = orders.dropna(subset=["order_delivered_customer_date", "order_purchase_timestamp"])
average_order_delivery_time = (valid_orders["order_delivered_customer_date"] - valid_orders["order_purchase_timestamp"]).mean()
average_order_delivery_time_in_days = round(average_order_delivery_time.total_seconds()/(60*60*24),2)
print(f"{average_order_delivery_time_in_days} days.")

12.56 days.


In [39]:
# Find orders that were delivered after the expected delivery date.
missed_expected_delivery_date = (orders["order_estimated_delivery_date"] - orders["order_delivered_customer_date"])
delayed_orders = missed_expected_delivery_date[missed_expected_delivery_date < pd.Timedelta(0) ]
delayed_orders.index = orders.loc[delayed_orders.index, "order_id"]
delayed_orders.dt.days.abs()

order_id
203096f03d82e0dffbc41ebc2e2bcfb7    12
fbf9ac61453ac646ce8ad9783d7d0af6    10
8563039e855156e48fccee4d611a3196     1
6ea2f835b4556291ffdc53fa0b3b95e8     8
66e4624ae69e7dc89bd50222b59f581f     2
                                    ..
b3112ca67f3afd4e20cf2ee91fc4f804     6
0fa1fab1d7c1211c824596ed5e111e3c     4
38e9133ce29f6bbe35aed9c3863dce01    22
d692ef54145c9cb3322ec2e5508aa3f4     3
cfa78b997e329a5295b4ee6972c02979     9
Length: 7827, dtype: int64

In [40]:
# Find Number of delayed orders.
print(f"Number of delayed orders : {delayed_orders.count()}")

Number of delayed orders : 7827


In [41]:
# Calculate the Percentage of delayed orders.
delayed_orders_percentage = round((delayed_orders.count()/ orders["order_id"].count()) * 100,2)
print(f"Percentage of delayed orders : {delayed_orders_percentage} %")

Percentage of delayed orders : 7.87 %


# Payments analysis

## 01-Payment value analysis

In [42]:
# Find Total value of payments.
total_payments = round(payments["payment_value"].sum(),2)
print(f"Total value of payments : {total_payments} BRL")

Total value of payments : 16008872.12 BRL


In [43]:
# Find Average of payments value.
Average_of_payments = round(payments["payment_value"].mean(),2)
print(f"Average value of payments : {Average_of_payments} BRL") 

Average value of payments : 154.1 BRL


In [44]:
# Find max payments value.
max_payment = round(payments["payment_value"].max(),2)
print(f"Max value of payment : {max_payment} BRL")

Max value of payment : 13664.08 BRL


In [45]:
# Find min payments value.
min_payment = round(payments["payment_value"].min(),2)
print(f"Min value of payment : {min_payment} BRL")

Min value of payment : 0.0 BRL


In [46]:
# Find most payment type.
most_payment_type = payments["payment_type"].value_counts().idxmax()
print(f"Most payment type : {most_payment_type}")

Most payment type : credit_card


In [47]:
# Find least payment type.
least_payment_type = payments["payment_type"].value_counts().idxmin()
print(f"least payment type : {least_payment_type}")

least payment type : not_defined


In [48]:
# Find the least used payment type, excluding "not_defined" type.
payments_excluding_not_defined = payments[payments["payment_type"] != "not_defined"]
least_used_payment_type = payments_excluding_not_defined["payment_type"].value_counts().idxmin()
print(f"least payment type excluding not defined type : {least_used_payment_type}")

least payment type excluding not defined type : debit_card


In [49]:
# Count the number of payments for each payment type, excluding "not_defined" type.
payments_excluding_not_defined["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
Name: count, dtype: int64

In [50]:
# calculate the precentage of each payment type, excluding "not_defined" type.
round(payments_excluding_not_defined["payment_type"].value_counts(normalize=True)*100,2)

payment_type
credit_card    73.92
boleto         19.04
voucher         5.56
debit_card      1.47
Name: proportion, dtype: float64

In [51]:
# Total value of payments for each payment type sorted in descending order.
payments.groupby("payment_type")["payment_value"].sum().sort_values(ascending=False)

payment_type
credit_card    12542084.19
boleto          2869361.27
voucher          379436.87
debit_card       217989.79
not_defined           0.00
Name: payment_value, dtype: float64

In [52]:
# Find average of payment value for each payment type sorted in descending order.
round(payments.groupby("payment_type")["payment_value"].mean().sort_values(ascending=False),2)

payment_type
credit_card    163.32
boleto         145.03
debit_card     142.57
voucher         65.70
not_defined      0.00
Name: payment_value, dtype: float64

## 02-Payment installments analysis

In [53]:
# Find average of payment installments.
average_of_payment_installments = round(payments["payment_installments"].mean(),2)
print(f"The average of payment installments : {average_of_payment_installments}")

The average of payment installments : 2.85


In [54]:
# Find max payment installments.
max_of_payment_installments = payments["payment_installments"].max()
print(f"The most value of payment installments : {max_of_payment_installments}")

The most value of payment installments : 24


In [55]:
# Find min payment installments.
min_of_payment_installments = payments["payment_installments"].min()
print(f"The least value of payment installments : {min_of_payment_installments}")

The least value of payment installments : 0


In [56]:
payments[payments["payment_installments"] == 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [57]:
# Find The most commonly used payment installment.
most_commonly_used_payment_installment = payments["payment_installments"].value_counts().idxmax()
print(f"The most commonly used payment installment : {most_commonly_used_payment_installment}")

The most commonly used payment installment : 1


In [58]:
# Find The least commonly used payment installment.
least_commonly_used_payment_installment = payments["payment_installments"].value_counts().idxmin()
print(f"The least commonly used payment installment : {least_commonly_used_payment_installment}")

The least commonly used payment installment : 22


In [59]:
# count the number of orders in each payment installments.
number_of_payment_installments = payments["payment_installments"].value_counts()
number_of_payment_installments.sort_index()

payment_installments
0         2
1     52546
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64

In [60]:
# count the precentage of orders in each payment installments.
precentage_of_payment_installments = round(payments["payment_installments"].value_counts(normalize=True)*100,3)
precentage_of_payment_installments.sort_index()

payment_installments
0      0.002
1     50.580
2     11.949
3     10.070
4      6.832
5      5.043
6      3.773
7      1.565
8      4.108
9      0.620
10     5.129
11     0.022
12     0.128
13     0.015
14     0.014
15     0.071
16     0.005
17     0.008
18     0.026
20     0.016
21     0.003
22     0.001
23     0.001
24     0.017
Name: proportion, dtype: float64

Insight 1 — Payment Preference
Credit card is the dominant payment method, indicating a strong customer preference for card-based payments.

Insight 2 — Payment Revenue
Credit card generates the highest total payment value by a significant margin.

Insight 3 — Average Transaction Value
Credit card also has the highest average payment value among the defined payment methods.

Insight 4 — Installment Behavior
Single-payment transactions represent about 50.58% of payment records, making one installment the most common payment option.

Insight 5 — Voucher Usage
Voucher payments have a noticeably lower average payment value (65.70) compared with other defined payment methods.

# Reviews analysis

## 01-Review Score Analysis

In [61]:
# Find average of review score.
average_of_review_score = round(reviews["review_score"].mean(),2)
print(f"The average of review score : {average_of_review_score}")

The average of review score : 4.09


In [62]:
# Find max review score.
print(f"The most review score : {reviews["review_score"].max()}")

The most review score : 5


In [63]:
# Find min review score.
print(f"The least review score : {reviews["review_score"].min()}")

The least review score : 1


In [64]:
# Find number of ratings for each score.
reviews["review_score"].value_counts(sort=True)

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

In [65]:
# calculate the precentage of review score.
round(reviews["review_score"].value_counts(normalize=True)*100,2)

review_score
5    57.78
4    19.29
1    11.51
3     8.24
2     3.18
Name: proportion, dtype: float64

In [66]:
# Find The most commonly used rating.
print(f"The most commonly used rating :{reviews["review_score"].value_counts().idxmax()}")

The most commonly used rating :5


In [67]:
# Find The least commonly used rating.
print(f"The least commonly used rating :{reviews["review_score"].value_counts().idxmin()}")

The least commonly used rating :2


In [68]:
# calculate the percentage of positive ratings.
rating_percentage = round(reviews["review_score"].value_counts(normalize=True)*100,2)
positive = rating_percentage[rating_percentage.index.isin(values=[4,5])].sum()
print(f"The percentage of positive rating : {positive} %")

The percentage of positive rating : 77.07 %


In [69]:
# calculate the percentage of neutral ratings.
neutral = rating_percentage[rating_percentage.index == 3].sum()
print(f"The percentage of neutral rating : {neutral} %")

The percentage of neutral rating : 8.24 %


In [70]:
# calculate the percentage of negative ratings.
negative = rating_percentage[rating_percentage.index.isin(values=[1,2])].sum()
print(f"The percentage of negative rating : {negative} %")

The percentage of negative rating : 14.69 %


## 02-Review comment analysis

In [71]:
# Find how many reviews actually have comments.
print(f"Reviews actually have comment : {reviews["review_comment_message"].count()}")

Reviews actually have comment : 40977


In [72]:
# Find how many reviews have no comments.
print(f"Reviews have no comments : {reviews["review_comment_message"].isna().sum()}")

Reviews have no comments : 58247


In [73]:
# Calculate the precentage of how many reviews actually have comments.
precentage_of_reviews_actually_have_comments = round((reviews["review_comment_message"].count()/reviews["review_id"].count())*100,2)
print(f"The precentage of how many reviews actually have comments : {precentage_of_reviews_actually_have_comments} %")

The precentage of how many reviews actually have comments : 41.3 %


In [74]:
# Calculate the precentage of how many reviews have no comments.
precentage_of_reviews_have_no_comments = round((reviews["review_comment_message"].isna().sum()/reviews["review_id"].count())*100,2)
print(f"The precentage of how many reviews have no comments : {precentage_of_reviews_have_no_comments} %")

The precentage of how many reviews have no comments : 58.7 %


In [75]:
# Find number of review score that include comments.
reviews.groupby("review_score")["review_comment_message"].count()

review_score
1     8745
2     2145
3     3557
4     5976
5    20554
Name: review_comment_message, dtype: int64

In [76]:
# The percentage of reviews containing comments within each review score.
review_score_that_include_comments = reviews.groupby("review_score")["review_comment_message"].count()
number_of_reviews_of_review_score = reviews.groupby("review_score")["review_id"].count()
round((review_score_that_include_comments/number_of_reviews_of_review_score)*100,2)

review_score
1    76.55
2    68.07
3    43.49
4    31.22
5    35.85
dtype: float64

Insight-1

The results indicate that customers who leave negative ratings are more likely to write comments
than those who leave positive ratings.

# Geographic Analysis

In [77]:
# Count the number of customers in each state.
customers["customer_state"].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [78]:
# Count the number of orders in each state.
orders_customers_orderItems["customer_state"].value_counts()

customer_state
SP    47820
RJ    14669
MG    13220
RS     6269
PR     5787
SC     4201
BA     3821
DF     2421
GO     2346
ES     2264
PE     1810
CE     1487
PA     1085
MT     1059
MA      831
MS      825
PB      606
PI      544
RN      532
AL      446
SE      390
TO      316
RO      284
AM      166
AC       92
AP       82
RR       52
Name: count, dtype: int64

Insight

São Paulo has the highest number of customers and orders, making it the strongest market geographically.
Roraima has the lowest number of customers and orders, indicating relatively low.

In [79]:
# Find the total sales in each state.
round(orders_customers_orderItems.groupby("customer_state")["price"].sum().sort_values(ascending=False),2)

customer_state
SP    5202955.05
RJ    1824092.67
MG    1585308.03
RS     750304.02
PR     683083.76
SC     520553.34
BA     511349.99
DF     302603.94
GO     294591.95
ES     275037.31
PE     262788.03
CE     227254.71
PA     178947.81
MT     156453.53
MA     119648.22
MS     116812.64
PB     115268.08
PI      86914.08
RN      83034.98
AL      80314.81
SE      58920.85
TO      49621.74
RO      46140.64
AM      22356.84
AC      15982.95
AP      13474.30
RR       7829.43
Name: price, dtype: float64

In [80]:
# Find the percentage of total sales in each state.
state_sales = orders_customers_orderItems.groupby("customer_state")["price"].sum()
state_sales_percentages = round(state_sales / orders_customers_orderItems["price"].sum() * 100, 2).sort_values(ascending=False)
state_sales_percentages

customer_state
SP    38.28
RJ    13.42
MG    11.66
RS     5.52
PR     5.03
SC     3.83
BA     3.76
DF     2.23
GO     2.17
ES     2.02
PE     1.93
CE     1.67
PA     1.32
MT     1.15
MA     0.88
MS     0.86
PB     0.85
PI     0.64
RN     0.61
AL     0.59
SE     0.43
TO     0.37
RO     0.34
AM     0.16
AC     0.12
AP     0.10
RR     0.06
Name: price, dtype: float64

In [81]:
# Find the top 10 cities with the highest number of orders.
orders_customers_orderItems["customer_city"].value_counts().head(10)

customer_city
sao paulo                17946
rio de janeiro            7885
belo horizonte            3167
brasilia                  2407
curitiba                  1762
campinas                  1669
porto alegre              1619
salvador                  1419
guarulhos                 1340
sao bernardo do campo     1070
Name: count, dtype: int64

In [82]:
# Find the top 10 cities with the lowest number of orders.
orders_customers_orderItems["customer_city"].value_counts().tail(10)

customer_city
agua fria de goias        1
oiapoque                  1
caldas brandao            1
dias d avila              1
desterro de entre rios    1
agua nova                 1
umburanas                 1
messias targino           1
campo do tenente          1
nova vicosa               1
Name: count, dtype: int64

In [83]:
# Find the average sales in each state.
round(orders_customers_orderItems.groupby("customer_state")["price"].sum().mean(),2)

np.float64(503394.21)

In [84]:
# Find the average freight value in each state.
round(orders_customers_orderItems.groupby("customer_state")["freight_value"].sum().mean(),2)

np.float64(83404.06)

In [85]:
# Find the average delivery time from the carrier to the customer in each state excluding orders with missing delivered customer date or missing delivered carrier date.
orders_customers_orderItems = orders_customers_orderItems.dropna(subset=["order_delivered_customer_date", "order_delivered_carrier_date"])
orders_customers_orderItems["average_carrier_to_customer_delivery_time"] = orders_customers_orderItems["order_delivered_customer_date"] - orders_customers_orderItems["order_delivered_carrier_date"]
state_average_delivery_time = orders_customers_orderItems.groupby("customer_state")["average_carrier_to_customer_delivery_time"].mean().sort_values(ascending=True)
round((state_average_delivery_time.dt.total_seconds())/(60*60*24),2).sort_values(ascending=True)

customer_state
SP     5.52
PR     8.62
MG     8.71
DF     9.68
SC    11.59
RJ    11.76
RS    11.90
ES    12.20
GO    12.30
MS    12.32
TO    14.00
MT    14.77
PE    14.97
RN    15.63
BA    15.86
PI    16.18
RO    16.88
PB    16.94
AC    17.32
CE    17.61
MA    17.77
SE    17.78
PA    20.30
AL    20.84
AM    23.51
AP    24.56
RR    24.57
Name: average_carrier_to_customer_delivery_time, dtype: float64

Insight

São Paulo has the shortest average carrier-to-customer delivery time, while Roraima has the longest,
highlighting significant geographic differences in delivery performance.

In [86]:
# Find the average order value for each state.
total_sales_per_order =  orders_customers_orderItems.groupby("order_id")["price"].sum()
state_per_order = orders_customers_orderItems.groupby("order_id")["customer_state"].first()
order_summary = pd.DataFrame({"total_sales": total_sales_per_order, "customer_state": state_per_order})
round(order_summary.groupby("customer_state")["total_sales"].mean().sort_values(ascending=False),2)

customer_state
PB    217.77
AP    199.62
AC    199.14
AL    198.63
RO    187.99
PA    184.43
PI    177.99
TO    176.65
RN    173.22
RR    172.13
CE    171.82
MT    171.77
SE    168.88
MS    164.66
MA    163.19
PE    158.12
AM    152.80
BA    151.59
GO    144.53
SC    142.98
DF    142.55
RJ    142.48
MG    136.72
RS    136.36
PR    135.30
ES    134.66
SP    125.12
Name: total_sales, dtype: float64

insight1

Paraíba has the highest average order value at 215.05, while São Paulo has the lowest at 124.63. 
Despite São Paulo being the strongest state in terms of customer and order volume, its customers have the lowest average order value.

Insight2

São Paulo contributes 38.28% of total sales, making it by far the largest revenue-generating state.
Rio de Janeiro and Minas Gerais follow with 13.42% and 11.66%, respectively. Together, these three states account for 63.36% of total sales, indicating a strong concentration of revenue in Brazil’s leading markets.